# Bitcoin Price Prediction Project
This notebook contains the complete workflow for predicting Bitcoin prices using Macroeconomic indicators.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# ============ A) Drive / Paths ============
# Use local directory where the notebook is located
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "data", "merged_with_lags.csv")

RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR  = os.path.join(BASE_DIR, "models")
PLOTS_DIR   = os.path.join(BASE_DIR, "plots")
NOTEBOOKS_DIR = os.path.join(BASE_DIR, "notebooks")

os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(NOTEBOOKS_DIR, exist_ok=True)

NAME_TAG = "diyor"
MODELS_OUT_DIR = os.path.join(MODELS_DIR, f"models_advanced_{NAME_TAG}")
PLOTS_OUT_DIR  = os.path.join(PLOTS_DIR,  f"plots_advanced_{NAME_TAG}")
os.makedirs(MODELS_OUT_DIR, exist_ok=True)
os.makedirs(PLOTS_OUT_DIR, exist_ok=True)

RESULTS_CSV = os.path.join(RESULTS_DIR, f"results_advanced_{NAME_TAG}.csv")

In [ ]:
# ============ Helper: Metrics ============
def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mape_percent(y_true, y_pred)
    return rmse, mae, r2, mape

In [ ]:
# ============ ADIM A) Load & sort ============
print(f"Loading data from: {DATA_PATH}")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dosya bulunamadı: {DATA_PATH}. Lütfen 'merged_with_lags.csv' dosyasının data klasöründe olduğundan emin olun.")

df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

targets = ["Target_1d","Target_7d","Target_30d","Target_365d"]
df.head()

In [ ]:
# ============ ADIM B) X prepare ============
X = df.drop(columns=["Date"] + targets)
X = X.select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan).ffill().bfill()

# ============ ADIM C) Split ============
test_ratio = 0.20
split_idx = int(len(df) * (1 - test_ratio))

X_train = X.iloc[:split_idx].copy()
X_test  = X.iloc[split_idx:].copy()
dates_test = df["Date"].iloc[split_idx:].reset_index(drop=True)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
# ============ TimeSeries CV (BONUS) ============
tscv = TimeSeriesSplit(n_splits=5)

# ============ Model Pipelines + Tuning Spaces ============
# Pipeline: SelectKBest(mutual_info_regression) + Model

hgb_pipe = Pipeline(steps=[
    ("kbest", SelectKBest(score_func=mutual_info_regression, k=50)),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

ada_pipe = Pipeline(steps=[
    ("kbest", SelectKBest(score_func=mutual_info_regression, k=50)),
    ("model", AdaBoostRegressor(random_state=42))
])

# RandomizedSearchCV param grids
# Not: k değeri feature sayısından büyük olamaz; biz aralığı güvenli tuttuk.
hgb_params = {
    "kbest__k": [30, 50, 80, 120],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [None, 3, 5, 8],
    "model__max_iter": [200, 400, 800],
    "model__min_samples_leaf": [10, 20, 50]
}

ada_params = {
    "kbest__k": [30, 50, 80, 120],
    "model__n_estimators": [100, 300, 600],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
    "model__loss": ["linear", "square", "exponential"]
}

search_hgb = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=hgb_params,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_ada = RandomizedSearchCV(
    estimator=ada_pipe,
    param_distributions=ada_params,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

models_to_run = [
    ("HistGradientBoosting", search_hgb),
    ("AdaBoost", search_ada)
]

In [ ]:
# ============ ADIM D/E/F/G/H) Loop targets/models ============
results_rows = []

for target in targets:
    print("\n==============================")
    print("TARGET:", target)
    print("==============================")

    y = df[target].replace([np.inf, -np.inf], np.nan).ffill().bfill()
    y_train = y.iloc[:split_idx].copy()
    y_test  = y.iloc[split_idx:].copy().reset_index(drop=True)

    for model_name, search in models_to_run:
        print(f"\n--- Model: {model_name} | Tuning: RandomizedSearchCV (TimeSeriesSplit) ---")

        # Fit search
        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        best_params = search.best_params_

        # Predict
        pred = best_model.predict(X_test)

        # Metrics
        rmse, mae, r2, mape = compute_metrics(y_test, pred)

        # Plot (Real vs Pred)
        plt.figure(figsize=(12,5))
        plt.plot(dates_test, y_test, label="Real")
        plt.plot(dates_test, pred, label="Pred")
        plt.title(f"{model_name} - {target} (Real vs Pred)")
        plt.xlabel("Date")
        plt.ylabel(target)
        plt.legend()
        plt.tight_layout()
        plt.show()

        plot_path = os.path.join(PLOTS_OUT_DIR, f"plot_{model_name}_{target}.png")
        plt.savefig(plot_path, dpi=150)
        plt.close()

        # Save model (joblib) + feature cols
        model_path = os.path.join(MODELS_OUT_DIR, f"{model_name}_{target}.joblib")
        joblib.dump(
            {
                "model": best_model,
                "feature_cols": X.columns.tolist(),
                "target": target,
                "best_params": best_params
            },
            model_path
        )

        # kbest chosen features count
        chosen_k = best_model.named_steps["kbest"].k

        # Results row
        results_rows.append({
            "Target": target,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "MAPE_%": mape,
            "Num_Features": int(chosen_k),
            "Best_Params": str(best_params)
        })

        print(f"RMSE={rmse:.4f} | MAE={mae:.4f} | R2={r2:.4f} | MAPE={mape:.2f}%")
        print("Saved:", model_path)
        print("Plot :", plot_path)

# Save results CSV
results_df = pd.DataFrame(results_rows)
try:
    results_df.to_csv(RESULTS_CSV, index=False)
    print("\n✅ RESULTS SAVED:", RESULTS_CSV)
except PermissionError:
    import time
    timestamp = int(time.time())
    new_csv_path = RESULTS_CSV.replace(".csv", f"_{timestamp}.csv")
    print(f"\n⚠️ UYARI: '{RESULTS_CSV}' dosyası başka bir programda açık olduğu için yazılamadı.")
    print(f"✅ Sonuçlar şuraya kaydedildi: {new_csv_path}")
    results_df.to_csv(new_csv_path, index=False)

# Show top models per target (lowest RMSE)
print("\n" + "="*60)
print("   🏆 BEST MODELS PER TARGET (Sorted by RMSE)   ")
print("="*60)

best_df = results_df.sort_values(["Target", "RMSE"]).groupby("Target").head(1)

# Select and reorder columns
cols_to_show = ["Target", "Model", "RMSE", "R2", "MAE", "MAPE_%"]

# Print formatted
print(best_df[cols_to_show].to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))
print("="*60)